In [3]:
from importlib import reload ,import_module
import module.utilize as utilize
import module.multiVariant as multiVariant
import module.singleVariant as singleVariant
import module.multiHistogramBase as multiHistogramBase
import numpy as np
from numba import njit,jit, float32
import module.singleVariantCopulaBase as CopulaBase
from tqdm import tqdm
import time
from multiprocessing import Pool
from sklearn.metrics import root_mean_squared_error
import cupy as cp
import module.multiHistogramSparse as multiHistogramSparse
reload(utilize)
reload(multiVariant)
reload(singleVariant)
reload(multiHistogramBase)
reload(CopulaBase)
reload(multiHistogramSparse)
import module.ananlysisFuncion as analysisFunction


startTime=time.time()

#attribute_names=np.array(["phi_grav","particle_mass_density","zmom","ymom"])
attribute_names=np.array(["phi_grav","particle_mass_density"])
incremental_number=300
all_ensamble_data=np.empty([0,incremental_number,64,64,64])

for name in attribute_names:
    data=utilize.readFiles(name,incremental_number)
    data=data.reshape(1,incremental_number,64,64,64)
    all_ensamble_data=np.append(all_ensamble_data,data,axis=0)

#print(all_ensamble_data.shape)
#print(all_ensamble_data[0].shape)
covBlockSize=5
dataBlockSize=6
binsNumber=128
sizeZ=60
sizeY=60
sizeX=60
minMaxBlockSize=2
isMinMax=False

conditions=np.array([[0,1e5],[3e10,4e10]])

print("start fit model")
with tqdm(total=1, desc="Model fitting") as pbar:

    oursModel=multiVariant.multiDistCopula3D.load(f"Nyx_{attribute_names.shape[0]}varaibles_{incremental_number}members_{binsNumber}Bins_dBlock5_cBlock5")
    print("ours complete fit")
    pbar.update(1)


result=np.zeros([sizeZ,sizeY,sizeX],dtype=np.float32)
oursResult=np.zeros([sizeZ,sizeY,sizeX],dtype=np.float32)


with tqdm(total=sizeZ*sizeY*sizeX, desc="總進度") as pbar:
    for idx in range(sizeZ * sizeY * sizeX):
        
        z = idx // (sizeY * sizeX)
        y = (idx // sizeX) % sizeY
        x = idx % sizeX      

        #Ground Truth
        samples=all_ensamble_data[:, :, z, y, x]
        samples=samples.T
        prob=analysisFunction.probability_in_range_numba(samples,conditions)
        result[z,y,x]=prob

        #Ours
        samples=oursModel.sampleByPos(z,y,x)
        prob=analysisFunction.probability_in_range_numba(samples,conditions)
        oursResult[z,y,x]=prob
        
        

        pbar.update(1)

result.tofile(f"NyxGT_{attribute_names.shape[0]}varaibles.bin")
oursResult.tofile(f"NyxOus_{attribute_names.shape[0]}varaibles_DBlock{dataBlockSize}_{binsNumber}bin.bin")


start fit model


Model fitting: 100%|██████████| 1/1 [00:00<00:00, 23.60it/s]


ours complete fit


總進度: 100%|██████████| 216000/216000 [12:26<00:00, 289.28it/s]


In [4]:
oursModel=multiVariant.multiDistCopula3D.load(f"Nyx_{attribute_names.shape[0]}varaibles_{incremental_number}members_{binsNumber}Bins_dBlock5_cBlock5")
oursModel.getStorageSize()

v0: 357480
v1: 110016
cov:10368.0
sum:477864.0
